In [1]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import numpy as np
import random
from geopy.distance import geodesic
import geopandas as gpd
import csv

In [2]:
#Reading graphml
G = nx.read_graphml(r"#######provide the path of the graphml file#####")

In [3]:
#Creating a file to store all outputs
f1 = open("config_random.txt","w")

In [4]:
#Checking the number of nodes and edges
print("num of nodes = ", G.number_of_nodes())
print("num of edges = ", G.number_of_edges())

#Storing output in a file
f1.write("Number of nodes:" + str(G.number_of_nodes()) + "\n")
f1.write("Number of edges:" + str(G.number_of_edges()) + "\n\n")

num of nodes =  841
num of edges =  1183


22

In [5]:
#Create a dict of degree sequences
degree_sequences = dict(G.degree())

#Create a list of degree sequences
degree_list = list(degree_sequences.values())

In [6]:
#Getting unique degrees and their frequencies
freq_degrees = list(set(degree_list))

#Print the frequency of each degree
f1.write(f"Frequency of the degree distribution for the empirical network is: \n")

for degree in freq_degrees:
    frequency = degree_list.count(degree)
    print(f"Degree {degree}: {frequency} nodes")
    f1.write(f"Degree {degree}: {frequency} nodes\n")

f1.write("\n\n")

Degree 1: 127 nodes
Degree 2: 14 nodes
Degree 3: 591 nodes
Degree 4: 107 nodes
Degree 5: 2 nodes


2

In [7]:
#Extracting x and y attributes for all nodes
x_attributes = nx.get_node_attributes(G, 'x')
y_attributes = nx.get_node_attributes(G, 'y')

In [8]:
#Combining the data into list of tuples
data_node = list(zip(x_attributes.keys(),x_attributes.values(),y_attributes.keys(),y_attributes.values()))

#CSV file path 
csv_file_node = "node_attributes_random.csv"

#Writing data in CSV file
with open(csv_file_node, mode='w', newline='') as file:
    writer = csv.writer(file)
    
    #Writing header
    writer.writerow(['node_id_x', 'x_attribute','node_id_y', 'y_attribute'])
    
    #Write data
    writer.writerows(data_node)

print(f"Data has been successfully written to {csv_file_node}")

Data has been successfully written to dholera_node_attributes_random.csv


In [9]:
#Extracting weight attributes for all edges
weight_attributes = nx.get_edge_attributes(G, 'weight')


In [10]:
#Combine keys, x, and y attributes into a list of tuples
data_edge = list(zip(weight_attributes.keys(), weight_attributes.values()))

#Define the CSV file path
csv_file_edge = "edges_attributes_random.csv"

#Write the data to the CSV file
with open(csv_file_edge, mode='w', newline='') as file:
    writer = csv.writer(file)
    
    #Write header
    writer.writerow(['Edge_pairs', 'Edge_weight'])
    
    #Write data
    writer.writerows(data_edge)

print(f"Data has been successfully written to {csv_file_edge}")


Data has been successfully written to dholera_edges_attributes_random.csv


In [11]:
#Calculate the average weighted global clustering coefficient
wg_cc_G = nx.average_clustering(G, weight="weight")
print("The average clustering coefficient of the network is: ", wg_cc_G)

#Storing output in a file
f1.write(f"The average clustering coefficient of the network is: {wg_cc_G} \n")

The average clustering coefficient of the network is:  0.0013598038811683667


77

In [12]:
#Calculating the pairs of all shortest path length from dijkstra algorithm 
shortest_path_lengths_dict = dict(nx.all_pairs_dijkstra_path_length(G, weight='weight'))

In [13]:
#Calculating the sum of inverse for all values in all nested dict
inner_inverse_sum = 0
for outer_key, inner_dict in shortest_path_lengths_dict.items():
    for inner_values in inner_dict.values():
        if inner_values != 0:
            values = 1/inner_values
            inner_inverse_sum += values

In [14]:
#calculating the value of n which should be equal to number of nodes here. 
n= len(shortest_path_lengths_dict)
print("Number of nodes: ", n)
#Calculating Number of pairs = n(n-1) not n(n-1)/2 because in dict we are considering twice
num_pairs = n*(n-1)
print("Number of pairs: ",num_pairs)

#Storing output in a file
f1.write(f"Number of nodes: {n} \n")
f1.write(f"Number of pairs: {num_pairs} \n")

Number of nodes:  841
Number of pairs:  706440


25

In [15]:
#Calculating the efficiency of the network
Efficiency_G = (1/num_pairs)*inner_inverse_sum
print("Efficiency of the network is:", Efficiency_G)

#Storing output in a file
f1.write(f"Efficiency of the network is: {Efficiency_G} \n")

Efficiency of the network is: 0.3302645771910532


50

In [16]:
#calculating the value of n which should be equal to number of nodes here. 
n= len(shortest_path_lengths_dict)
print("Number of nodes: ", n)
#Calculating Number of pairs = n(n-1) not n(n-1)/2 because in dict we are considering twice
num_pairs = n*(n-1)
print("Number of pairs: ",num_pairs)

Number of nodes:  841
Number of pairs:  706440


In [17]:
#Making ensembles of the configuration model
num_ensembels = 200
seed_val = range(180, num_ensembels + 1)

#Specifying the lists for calculating the clustering and efficiency of the configuration model
cc_config = []
efficiency_config = []
config_models = []

for seed in seed_val:
    #Create a configuration model
    config_model = nx.configuration_model(degree_list, create_using=nx.Graph(), seed=seed)

    #Assign 'x' and 'y' attributes to nodes in the configuration model
    for node in config_model.nodes():
        node_id = int(node)  #Convert the node identifier to an integer
        config_model.nodes[node]['x'] = x_attributes[str(node_id)]
        config_model.nodes[node]['y'] = y_attributes[str(node_id)]

    #Calculate geodesic distance between nodes and assign as edge weights
    for edge in config_model.edges():
        node1, node2 = edge
        x1, y1 = config_model.nodes[node1]['x'], config_model.nodes[node1]['y']
        x2, y2 = config_model.nodes[node2]['x'], config_model.nodes[node2]['y']

        #Calculate geodesic distance and assign as edge weight
        distance = geodesic((y1, x1), (y2, x2)).kilometers
        config_model[edge[0]][edge[1]]['weight'] = distance

    #Calculating the average clustering for the configuration model
    cc_config.append(nx.average_clustering(config_model, weight='weight'))

    #Calculating the Efficiency for the configuration model
    #Calculating the pairs of all shortest path length from Dijkstra algorithm
    shortest_path_lengths_dict = dict(nx.all_pairs_dijkstra_path_length(config_model, weight='weight'))

    #Calculating the sum of inverse for all values in all nested dict
    inner_inverse_sum = 0
    for outer_key, inner_dict in shortest_path_lengths_dict.items():
        for inner_values in inner_dict.values():
            if inner_values != 0:
                values = 1 / inner_values
                inner_inverse_sum += values

    #Calculating the efficiency of the network
    Efficiency_config = (1 / num_pairs) * inner_inverse_sum
    efficiency_config.append(Efficiency_config)

    #Append the current configuration model to the list
    config_models.append(config_model)
    
    
#Iterate through all configuration models
for i, config_model in enumerate(config_models):
    #Create a dict of degree sequences for the current model
    degree_sequences = dict(config_model.degree())

    #Create a list of degree sequences for the current mode
    degree_list = list(degree_sequences.values())

    #Get unique degrees and their frequencies for the current model
    freq_degrees = list(set(degree_list))
   
    #Print and write degree distribution information for the current model
    f1.write(f" \n Degree distribution for Configuration Model {i}: \n")
    for degree in freq_degrees:
        frequency = degree_list.count(degree)
        #print(f'Degree {degree}: {frequency} nodes')
        f1.write(f"Degree {degree}: {frequency} nodes \n")

#Average clustering and efficiency for the configuration model
av_cc_config = np.mean(cc_config)
print("Average clustering coefficient for the configuration model is: ", av_cc_config)

av_efficiency_config = np.mean(efficiency_config)
print("Average Efficiency for the configuration model is: ", av_efficiency_config)

#Storing output in a file
f1.write(f"List for the clustering coefficient value for all configuration models: {cc_config} \n")
f1.write(f"List for the efficiency value for all configuration models: {efficiency_config} \n\n")

f1.write(f"Average clustering coefficient for the configuration model is: {av_cc_config} \n")
f1.write(f"Average Efficiency for the configuration model is: {av_efficiency_config} \n\n")

#Saving the configuration model into a graphml file
for i, config_model in enumerate(config_models):
    nx.write_graphml(config_model, f"config_model_{i}.graphml")


Average clustering coefficient for the configuration model is:  0.00026714410229529824
Average Efficiency for the configuration model is:  0.01784985587293687


In [18]:
#Calculating the ratio of Ce and Cc
Ce_Cc = wg_cc_G/av_cc_config
print("The Ratio of Ce and Cc is : ", Ce_Cc)

#Calculating the ratio of Ee and Ec
Ee_Ec = Efficiency_G/av_efficiency_config
print("The ratio of Ee and Ec is : ", Ee_Ec)

#Storing output in a file
f1.write(f"The Ratio of Ce and Cc is : {Ce_Cc} \n")
f1.write(f"The ratio of Ee and Ec is : {Ee_Ec} \n")

The Ratio of Ce and Cc is :  5.090151231058263
The ratio of Ee and Ec is :  18.502366604079146


48

In [19]:
f1.close()

In [20]:
#Creating a file to store all outputs
f2 = open("dholera_random.txt","w")

In [21]:
#Average Degree
total_degree = sum(dict(G.degree()).values())
av_degree = total_degree / len(G.nodes())
print("The average degree of the network is: ", av_degree)
f2.write(f"The average degree of the network is: {av_degree} \n")

#probability of connection
p = av_degree / (len(G.nodes()) - 1)
print("The Probability of the connection in the network is: ", p)
f2.write(f"The Probability of the connection in the network is: {p} \n\n")

The average degree of the network is:  2.8133174791914386
The Probability of the connection in the network is:  0.003349187475227903


76

In [22]:
#Making ensembles of the random model
num_ensembels_r = 200
seed_val_r = range(180, num_ensembels_r + 1)

#Specifying the lists for calculating the clustering and efficiency of the random model
cc_random = []
efficiency_random = []
random_networks= []

for seed in seed_val_r:
    #Creating a random model
    er_random_graph = nx.erdos_renyi_graph(n, p, seed=seed)

    # Assign 'x' and 'y' attributes to nodes in the configuration model
    for node in er_random_graph.nodes():
        node_id = int(node)  # Convert the node identifier to an integer
        er_random_graph.nodes[node]['x'] = x_attributes[str(node_id)]
        er_random_graph.nodes[node]['y'] = y_attributes[str(node_id)]

    # Calculate geodesic distance between nodes and assign as edge weights
    for edge in er_random_graph.edges():
        node1, node2 = edge
        x1, y1 = er_random_graph.nodes[node1]['x'], er_random_graph.nodes[node1]['y']
        x2, y2 = er_random_graph.nodes[node2]['x'], er_random_graph.nodes[node2]['y']
    
        # Calculate geodesic distance and assign as edge weight
        distance = geodesic((y1, x1), (y2, x2)).kilometers
        er_random_graph[edge[0]][edge[1]]['weight'] = distance
    
    #Calculating the average clustering for the configuration model
    cc_random.append(nx.average_clustering(er_random_graph, weight = 'weight'))
    
    
    #Calculating the Efficiency for the configuration model
    #Calculating the pairs of all shortest path length from dijkstra algorithm 
    shortest_path_lengths_dict = dict(nx.all_pairs_dijkstra_path_length(er_random_graph, weight='weight'))
    
    #Calculating the sum of inverse for all values in all nested dict
    inner_inverse_sum = 0
    for outer_key, inner_dict in shortest_path_lengths_dict.items():
        for inner_values in inner_dict.values():
            if inner_values != 0:
                values = 1/inner_values
                inner_inverse_sum += values
                
    #Calculating the efficiency of the network
    Efficiency_random = (1/num_pairs)*inner_inverse_sum
    efficiency_random.append(Efficiency_random)
    
    #Save the random networks
    random_networks.append(er_random_graph)
    
av_cc_random = np.mean(cc_random)
print("Average clustering coefficient for the random model is: ",av_cc_random)

av_efficiency_random = np.mean(efficiency_random)
print("Average Efficiency for the random model is: ",av_efficiency_random)

#Storing output in a file
f2.write(f"List for the clustering coefficient value for all random models: {cc_random} \n")
f2.write(f"List for the efficiency value for all random models: {efficiency_random} \n\n")

f2.write(f"Average clustering coefficient for the random model is: {av_cc_random} \n")
f2.write(f"Average Efficiency for the random model is: {av_efficiency_random} \n\n")

# Save all random networks as GraphML files
for j, random_network in enumerate(random_networks):
    nx.write_graphml(random_network, f"random_network_{j + 1}.graphml")


Average clustering coefficient for the random model is:  0.0007568724806854391
Average Efficiency for the random model is:  0.020287531001388397


In [23]:
#Calculating the ratio of Ce and Cc
Ce_Cr = wg_cc_G/av_cc_random
print("The Ratio of Ce(empirical clustering) and Cr(random clustering) is : ", Ce_Cr)

#Calculating the ratio of Ee and Ec
Ee_Er = Efficiency_G/av_efficiency_random
print("The Ratio of Ee(empirical efficiency) and Cr(random efficiency) is : ", Ee_Er)

#Storing output in a file
f2.write(f"The Ratio of Ce(empirical clustering) and Cr(random clustering) is : {Ce_Cr} \n")
f2.write(f"The ratio of Ee(empirical efficiency) and Er(random efficiency) is : {Ee_Er} \n")

The Ratio of Ce(empirical clustering) and Cr(random clustering) is :  1.7966089610457243
The Ratio of Ee(empirical efficiency) and Cr(random efficiency) is :  16.27919026560951


88

In [24]:
f2.close()